# 6. Data Preprocessing

## 1. Introduction

Data preprocessing prepares the dataset for Machine Learning
algorithms.

The main preprocessing operations in this project are:

- Train, Validation, and Test splitting.
- Missing value imputation.
- Feature scaling.
- Consistent transformation of unseen data.

A strict separation between training, validation, and test data
is required to prevent data leakage.

## 2. Data Splitting Strategy

The dataset will be divided into three subsets:

- Training Set: 64%
- Validation Set: 16%
- Test Set: 20%

The split will use stratification to preserve the proportion
of fraudulent and legitimate transactions.

The test set will remain untouched during model selection,
hyperparameter tuning, and threshold selection.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import (
    CLEANED_DATA_PATH,
    RANDOM_STATE,
    TARGET_COLUMN,
    TEST_SIZE,
)

df = pd.read_csv(CLEANED_DATA_PATH)

X = df.drop(columns=[TARGET_COLUMN])

y = df[TARGET_COLUMN]

print("Dataset Shape:", df.shape)

Dataset Shape: (284807, 31)


In [11]:
# First split: 80% development data, 20% test data.

X_development, X_test, y_development, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

# Second split: 80% training, 20% validation
# of the development data.

X_train, X_validation, y_train, y_validation = train_test_split(
    X_development,
    y_development,
    test_size=0.20,
    stratify=y_development,
    random_state=RANDOM_STATE
)

print("Training Set:", X_train.shape)

print("Validation Set:", X_validation.shape)

print("Test Set:", X_test.shape)

Training Set: (182276, 30)
Validation Set: (45569, 30)
Test Set: (56962, 30)


In [12]:
split_summary = pd.DataFrame({
    "Training": y_train.value_counts(normalize=True),
    "Validation": y_validation.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True),
}).fillna(0)

display(
    split_summary * 100
)

,Training,Validation,Test
Class,,,
0,99.827185,99.826637,99.827955
1,0.172815,0.173363,0.172045


## 3. Missing Value Imputation

Missing numerical values are handled using median imputation.

The imputer is fitted only on the training data.

The fitted imputer is then applied to validation and test data.

This prevents information from validation and test observations
from influencing the preprocessing parameters.

In [13]:
from src.feature_engineering import FeatureEngineer
from src.preprocessing import create_preprocessor

feature_engineer = FeatureEngineer()

preprocessor = create_preprocessor()

In [14]:
# Feature engineering is fitted on training data.

X_train_features = feature_engineer.fit_transform(
    X_train
)

# The same transformation is applied to validation and test.

X_validation_features = feature_engineer.transform(
    X_validation
)

X_test_features = feature_engineer.transform(
    X_test
)

# Fit preprocessing only on training data.

X_train_processed = preprocessor.fit_transform(
    X_train_features
)

# Transform validation and test without refitting.

X_validation_processed = preprocessor.transform(
    X_validation_features
)

X_test_processed = preprocessor.transform(
    X_test_features
)

print("Training Processed Shape:", X_train_processed.shape)

print("Validation Processed Shape:", X_validation_processed.shape)

print("Test Processed Shape:", X_test_processed.shape)

Training Processed Shape: (182276, 32)
Validation Processed Shape: (45569, 32)
Test Processed Shape: (56962, 32)


## 4. Feature Scaling Analysis

StandardScaler transforms numerical features using:

z = (x - mean) / standard_deviation

The mean and standard deviation are learned from training data.

Scaling is particularly important for algorithms such as:

- Logistic Regression
- Support Vector Machines
- K-Nearest Neighbors

Tree-based algorithms generally do not require feature scaling.

The final training pipeline will retain the preprocessing
steps to ensure consistent transformations during inference.

In [15]:
processed_feature_names = (
    feature_engineer.get_feature_names_out()
)

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

display(
    X_train_processed_df.describe().T.head(15)
)

,count,mean,std,min,25%,50%,75%,max
Time,182276.0,1.350326e-16,1.000003,-1.998772,-0.856697,-0.212280,0.936588,1.640602
V1,182276.0,5.769291e-18,1.000003,-28.819047,-0.470817,0.009809,0.671964,1.252385
V2,182276.0,5.496419e-18,1.000003,-44.249539,-0.366028,0.038543,0.488104,13.421537
V3,182276.0,-4.989657e-18,1.000003,-22.174532,-0.586724,0.119806,0.677041,6.178298
V4,182276.0,1.871121e-18,1.000003,-4.009376,-0.599270,-0.014935,0.527626,11.912145
V5,182276.0,9.433570e-18,1.000003,-30.942576,-0.506979,-0.038437,0.449644,25.551954
V6,182276.0,-6.704851e-18,1.000003,-19.800105,-0.580165,-0.206264,0.301759,17.055295
V7,182276.0,7.796339e-20,1.000003,-35.682590,-0.453591,0.033120,0.467035,30.044672
V8,182276.0,0.000000e+00,1.000003,-61.200262,-0.173990,0.019133,0.273175,16.372751
V9,182276.0,1.621638e-17,1.000003,-12.224818,-0.583562,-0.046738,0.545342,14.191283


## 5. Data Leakage Prevention

The following rules are enforced:

1. Split data before fitting preprocessing transformations.
2. Fit the feature preprocessing steps on training data only.
3. Never apply SMOTE to validation or test data.
4. Never use test data for hyperparameter tuning.
5. Never select the classification threshold using test data.

The complete transformation workflow will be integrated
into a Machine Learning pipeline during model training.

In [16]:
assert len(X_train) + len(X_validation) + len(X_test) == len(df)

assert X_train.index.intersection(X_validation.index).empty

assert X_train.index.intersection(X_test.index).empty

assert X_validation.index.intersection(X_test.index).empty

print("Data splitting verification passed.")

print("No overlap between training, validation, and test indexes.")

Data splitting verification passed.
No overlap between training, validation, and test indexes.
